# Agente de correlaciones y relaciones entre ETFs

Guía de uso del paquete `etf_agent`: correlaciones **históricas, YTD, 3M y 1M** y
búsqueda de **cualquier tipo de relación** entre los ETFs más significativos.

Universo por defecto: `SPY, QQQ, IWM, EFA, EEM, TLT, IEF, TIP, SGOV, LQD, HYG, UUP,
GLD, USO, DBC, XLK, XLF, XLE, XLU, XLP, SOXX, DRAM`.

> Este notebook funciona en **modo demo** (datos sintéticos) para que puedas ejecutarlo
> sin internet. Para datos reales, usa `fetch_prices()` (ver la última sección).


## 1. Instalación y carga del universo

In [ ]:
# pip install -r requirements.txt   (una sola vez)

import pandas as pd
from etf_agent import (
    DEFAULT_TICKERS, make_demo_prices, fetch_prices, log_returns,
    correlation_matrices, top_pairs, rolling_correlation, build_report,
)
from etf_agent import relationships as rel
from etf_agent.config import TICKER_INFO

pd.set_option("display.width", 160)
print(f"{len(DEFAULT_TICKERS)} ETFs en el universo:")
for t in DEFAULT_TICKERS:
    print(f"  {t:<6} {TICKER_INFO.get(t, {}).get('clase', '')}")

## 2. Datos

`make_demo_prices` genera precios sintéticos con estructura factorial realista
(mercado, tipos, materias primas, crédito, semis, dólar) e incluye de forma deliberada
relaciones lead-lag y cointegración, para que el agente tenga cosas que encontrar.

In [ ]:
prices = make_demo_prices(start="2015-01-01")   # sin internet
returns = log_returns(prices)

prices.tail()

In [ ]:
# Evolución comparada (base 100, escala logarítmica)
(prices / prices.iloc[0] * 100).plot(figsize=(12, 5), logy=True, legend=False,
                                    title="Evolución comparada (base 100)");

## 3. Correlaciones por ventana: histórico, YTD, 3M, 1M

Se calculan sobre **retornos logarítmicos diarios** con Pearson (lineal) y Spearman
(rangos, robusto a outliers).

In [ ]:
mats = correlation_matrices(prices, windows=["max", "ytd", "3m", "1m"])

# Matriz de correlación de los últimos 3 meses
mats[("3m", "pearson")].round(2)

In [ ]:
# Comparación rápita: misma pareja en distintas ventanas
import itertools

pares = list(itertools.islice(itertools.combinations(prices.columns, 2), 0, 6))
filas = []
for a, b in pares:
    fila = {"par": f"{a}-{b}"}
    for w in ["max", "ytd", "3m", "1m"]:
        fila[w] = round(float(mats[(w, "pearson")].loc[a, b]), 3)
    filas.append(fila)
pd.DataFrame(filas)

In [ ]:
# Top pares más correlacionados en el histórico
top_pairs(mats[("max", "pearson")], n=10)

## 4. Heatmaps

El módulo `report` incluye la función `heatmap` (también se guardan como PNG al
generar el informe).

In [ ]:
from etf_agent.report import heatmap

heatmap(mats[("ytd", "pearson")], title="Correlación YTD (Pearson)");

## 5. Correlación móvil y régimen

La correlación media del universo sube en las crisis: es un indicador de
"todo se mueve junto" (diversificación que desaparece).

In [ ]:
from etf_agent.correlation import average_rolling_correlation

avg = average_rolling_correlation(returns, window=63)
avg.plot(figsize=(12, 3.5), title="Correlación media del universo (63 sesiones)",
         ylabel="correlación media", legend=False);

In [ ]:
# Correlación móvil de dos ETFs concretos
rolling_correlation(returns, "SPY", "TLT", window=63).plot(
    figsize=(12, 3), title="Correlación móvil SPY vs TLT (63 sesiones)");

## 6. Lead-lag: ¿quién anticipa a quién?

Para cada pareja se busca el rezago (hasta ±N sesiones) con mayor correlación.
`leader` = el que se mueve antes.

In [ ]:
ll = rel.lead_lag_table(returns, max_lag=10)
ll.head(10)

## 7. Causalidad de Granger

H0: `cause` **no** mejora la predicción de `effect`. Ojo: con muchas series aparecen
falsos positivos; contrasta siempre con la fuerza económica de la relación.

In [ ]:
gc = rel.granger_table(returns, max_lag=5)
print(f"Pares significativos (p<0.05): {int(gc['significativo'].sum())} de {len(gc)}")
gc.head(10)

## 8. Cointegración: relaciones de largo plazo

Pares cuyo spread revierte a la media (estrategias de pares). Incluye `hedge_ratio`
(beta de la regresión de cointegración), `half_life_days` (velocidad de reversión) y
`spread_zscore` (cuán lejos está el spread ahora).

In [ ]:
co = rel.cointegration_table(prices)
print(f"Pares cointegrados (p<0.05): {int(co['cointegrada'].sum())}")
co.head(10)

In [ ]:
# Spread del par más cointegrado
par = co.iloc[0]
a, b = par["A"], par["B"]
spread = rel.spread_series(prices, a, b, hedge_ratio=float(par["hedge_ratio"]))
z = (spread - spread.mean()) / spread.std()
z.plot(figsize=(12, 3), title=f"z-score del spread {a}-{b}");

## 9. Exposición factorial (betas)

Regresión OLS de cada ETF sobre factores (por defecto mercado `SPY`, tipos `TLT` y
oro `GLD`). El `beta` mide la sensibilidad; `R²`, cuánto explica el factor.

In [ ]:
betas = rel.factor_betas(returns, factors=["SPY", "TLT", "GLD"])
betas.round(3)

## 10. Factores latentes (PCA) y agrupamiento

PCA encuentra los factores comunes que mueven a todos los ETFs; el clustering los agrupa
por similitud de comportamiento.

In [ ]:
pca = rel.pca_factors(returns, n_components=4)
print("Varianza explicada:", pca["explained_variance_ratio"].round(3).to_dict())
pca["loadings"].round(3)

In [ ]:
from etf_agent.correlation import correlation_matrix, clustered_order
from etf_agent.report import plot_dendrogram

corr = correlation_matrix(returns)
plot_dendrogram(corr, order=clustered_order(corr))
clusters = rel.cluster_etfs(corr, n_clusters=5)
clusters

## 11. Dependencia no lineal (información mutua)

La información mutua detecta relaciones que la correlación de Pearson no ve
(por ejemplo dependencias en forma de U o en las colas).

In [ ]:
mi = rel.mutual_information_matrix(returns, bins=8)
mi.round(3)

## 12. Regímenes de estrés

Compara la correlación con el benchmark en los días malos (peor 10%) y buenos
(mejor 10%). `asimetria > 0` = la correlación se dispara cuando el mercado cae.

In [ ]:
reg = rel.regime_correlation(returns, benchmark="SPY")
reg.round(3)

In [ ]:
# Drawdowns del benchmark superiores al 10%
dd = rel.drawdown_episodes(prices, benchmark="SPY")
dd

## 13. Volatilidad y riesgo

In [ ]:
vol = rel.volatility_table(returns)
vol["max_drawdown_%"] = vol["etf"].map(rel.max_drawdown(prices)).round(1)
vol.round(3)

## 14. Informe completo (Markdown + HTML + CSV + PNG)

`build_report` ejecuta todos los análisis de arriba y escribe un informe listo para
compartir.

In [ ]:
result = build_report(prices, windows=["max", "ytd", "3m", "1m"],
                       factors=["SPY", "TLT", "GLD"], outdir="reports", demo=True)
print(result["markdown"])
print(result["html"])

## 15. Con datos reales

`fetch_prices()` descarga de Yahoo Finance (con fallback a Stooq) y guarda un cache
CSV por ticker en `data/etf_cache/`. Todo lo demás es idéntico.

In [ ]:
# Necesita internet
# prices_real = fetch_prices()                      # universo completo
# prices_real = fetch_prices(tickers=["SPY","TLT","GLD","UUP","USO","EFA","EEM"])
# result = build_report(prices_real, outdir="reports")
#
# Desde la línea de comandos:
#   python -m etf_agent analyze
#   python -m etf_agent dashboard